# Introduction to Denoising Diffusion Implicit Generative Models

## Stephen Elston

## Overview

Though these exercises you will gain some basic understanding of [**denoising deffusion implicit models (DDIMs)**](https://arxiv.org/abs/2010.02502). DDIMs are image generative models which synthesize high quality images.  

In the example in this notebook we use the [Oxford Flowers 102](https://www.tensorflow.org/datasets/catalog/oxford_flowers102)
dataset to train a DDIM model to generate a diverse set of flower images. This dataset contains 8,000 images of a diverse set of flowers.

> Attribution: This notebook is based on a Keras example notebook [Denoising Diffusion Implicit Models](https://keras.io/examples/generative/ddim/) by [András Béres](https://www.linkedin.com/in/andras-beres-789190210), updated 2022/06/2.

You can see a Keras example of the [denoising diffusion probabilistic model (DDPM)](https://arxiv.org/abs/2006.11239) [**here**](https://keras.io/examples/generative/ddpm/).  

## Introduction

### What are diffusion models?

Recently, [denoising diffusion models](https://arxiv.org/abs/2006.11239), including [score-based generative models](https://arxiv.org/abs/1907.05600), have gained popularity as a powerful class of image generative models. Recent large diffusion models, such as [DALL-E 2](https://openai.com/dall-e-2/) and [Imagen](https://imagen.research.google/), have shown good text-to-image generation capability. A drawback is that they are slower to sample from, because they require multiple forward passes for generating an image.

Diffusion refers to the process of turning a structured signal (an image) into a noise latent step-by-step. By simulating diffusion, we can generate noisy images from our training images. We then train a neural network to denoise the images. Using the trained network we apply a reverse diffusion process to generate an image from the noise latent.

![diffusion process gif](https://i.imgur.com/dipPOfa.gif)

<div style="text-align: center;"> <b> Diffusion models are trained to denoise a pure noise latent to generate images </b> </div>

## Running the Example Code     

This notebook has been tested using [Google Pro+](https://colab.research.google.com/). On several tests in Colab this notebook executed in under 1 hour with no time outs. It is recommended that you configure your runtime to use a GPU and `High-RAM. To change the runtime environment from the Jupyter menu for which you can select your runtime, `Runtime -> Change runtime type`.   

<img src="../img/ColabChangeRuntime.png" alt="Drawing" style="width:500px; height:500px"/>
<center>Box for configuring Colab runtime environment</center>     

If you encounter problems, please reach out to the instructional staff.  

To run this notebook it may be necessary to upgrade the versions of TensorFlow and Keras. If you encounter version difficulties, uncomment and execute the following code. Otherwise, you can skip this step.   

In [ ]:
#!pip install tensorflow --upgrade
#!pip install keras --upgrade

Execute the code in the cell below to import the required packages.  

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import math
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds

import keras
from keras import layers
from keras import ops

print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)

## Hyperparameters

The code in the cell below defines key hypterparameters for this notebook Execute this code.  

In [ ]:
# data
dataset_name = "oxford_flowers102"
dataset_repetitions = 5
## change this variable to 50
num_epochs = 50 # train for at least 50 epochs for good results
image_size = 64
# KID = Kernel Inception Distance, see related section
kid_image_size = 75
kid_diffusion_steps = 5
plot_diffusion_steps = 20

# sampling
min_signal_rate = 0.02
max_signal_rate = 0.95

# architecture
embedding_dims = 32
embedding_max_frequency = 1000.0
widths = [32, 64, 96, 128]
block_depth = 2

# optimization
batch_size = 64
ema = 0.999
learning_rate = 1e-4
weight_decay = 1e-4

## Data pipeline

The official splits for the [Oxford Flowers 102](https://www.tensorflow.org/datasets/catalog/oxford_flowers102)
dataset are imbalanced, as most of the images are contained in the test split. The creates new splits (80% train, 20% validation) using the [Tensorflow Datasets slicing API](https://www.tensorflow.org/datasets/splits). Additionally, center crops are applied as preprocessing. Execute this code to create the preprocessed image with the train and validation splits. 

In [ ]:

def preprocess_image(data):
    # center crop image
    height = ops.shape(data["image"])[0]
    width = ops.shape(data["image"])[1]
    crop_size = ops.minimum(height, width)
    image = tf.image.crop_to_bounding_box(
        data["image"],
        (height - crop_size) // 2,
        (width - crop_size) // 2,
        crop_size,
        crop_size,
    )

    # resize and clip
    # for image downsampling it is important to turn on antialiasing
    image = tf.image.resize(image, size=[image_size, image_size], antialias=True)
    return ops.clip(image / 255.0, 0.0, 1.0)


def prepare_dataset(split):
    # the validation dataset is shuffled as well, because data order matters
    # for the KID estimation
    return (
        tfds.load(dataset_name, split=split, shuffle_files=True)
        .map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
        .cache()
        .repeat(dataset_repetitions)
        .shuffle(10 * batch_size)
        .batch(batch_size, drop_remainder=True)
        .prefetch(buffer_size=tf.data.AUTOTUNE)
    )


# load dataset
train_dataset = prepare_dataset("train[:80%]+validation[:80%]+test[:80%]")
val_dataset = prepare_dataset("train[80%:]+validation[80%:]+test[80%:]")


## Kernel inception distance

We use the [Kernel Inception Distance (KID)](https://arxiv.org/abs/1801.01401) image quality metric to evaluate the generated images. You can see more details
[here](https://keras.io/examples/generative/gan_ada/#kernel-inception-distance).

In this notebook, the images are evaluated at a low resolution in the feature map of the Inception network (75x75 instead of 299x299). The metric is only measured on the validation set for computational efficiency. We also limit the number of sampling steps at evaluation to 5 for the same reason. Since the dataset is relatively small, we pass over the train and validation splits multiple times per epoch. The KID estimation is noisy and compute-intensive, and is evaluated only after many iterations.     

Execute the code in the cell below to load the KID class.  

In [ ]:

@keras.saving.register_keras_serializable()
class KID(keras.metrics.Metric):
    def __init__(self, name, **kwargs):
        super().__init__(name=name, **kwargs)

        # KID is estimated per batch and is averaged across batches
        self.kid_tracker = keras.metrics.Mean(name="kid_tracker")

        # a pretrained InceptionV3 is used without its classification layer
        # transform the pixel values to the 0-255 range, then use the same
        # preprocessing as during pretraining
        self.encoder = keras.Sequential(
            [
                keras.Input(shape=(image_size, image_size, 3)),
                layers.Rescaling(255.0),
                layers.Resizing(height=kid_image_size, width=kid_image_size),
                layers.Lambda(keras.applications.inception_v3.preprocess_input),
                keras.applications.InceptionV3(
                    include_top=False,
                    input_shape=(kid_image_size, kid_image_size, 3),
                    weights="imagenet",
                ),
                layers.GlobalAveragePooling2D(),
            ],
            name="inception_encoder",
        )

    def polynomial_kernel(self, features_1, features_2):
        feature_dimensions = ops.cast(ops.shape(features_1)[1], dtype="float32")
        return (
            features_1 @ ops.transpose(features_2) / feature_dimensions + 1.0
        ) ** 3.0

    def update_state(self, real_images, generated_images, sample_weight=None):
        real_features = self.encoder(real_images, training=False)
        generated_features = self.encoder(generated_images, training=False)

        # compute polynomial kernels using the two sets of features
        kernel_real = self.polynomial_kernel(real_features, real_features)
        kernel_generated = self.polynomial_kernel(
            generated_features, generated_features
        )
        kernel_cross = self.polynomial_kernel(real_features, generated_features)

        # estimate the squared maximum mean discrepancy using the average kernel values
        batch_size = real_features.shape[0]
        batch_size_f = ops.cast(batch_size, dtype="float32")
        mean_kernel_real = ops.sum(kernel_real * (1.0 - ops.eye(batch_size))) / (
            batch_size_f * (batch_size_f - 1.0)
        )
        mean_kernel_generated = ops.sum(
            kernel_generated * (1.0 - ops.eye(batch_size))
        ) / (batch_size_f * (batch_size_f - 1.0))
        mean_kernel_cross = ops.mean(kernel_cross)
        kid = mean_kernel_real + mean_kernel_generated - 2.0 * mean_kernel_cross

        # update the average KID estimate
        self.kid_tracker.update_state(kid)

    def result(self):
        return self.kid_tracker.result()

    def reset_state(self):
        self.kid_tracker.reset_state()


> **Exercise 7-1:** Examine the foregoing code for computing the KID and answer these questions in one or a few sentences.
> 1. Explain why the [keras.applications.InceptionV](https://keras.io/api/applications/inceptionv3/) model defined in the `__init__` method uses the `include_top=False` argument.
> 2. Explain the meaning of the `kernel_generated` and `kernel_real` variables are computed in the `polynomial_kernel` method. *Hint:* The `@` operator represents a matrix multiplication, an inner product in this case.     

> **Answers:**
> 1.       
> 2.    

## Denoising Network   

A key component of a diffusion image generation is the denoising model. Here we will use a modification of the well-known [U-Net](https://arxiv.org/abs/1505.04597) architecture for the learnable denoising model. The U-NET has several advantages for this purpose.    

* The U-NET processes learns the noise at multiple scales simultaneously.
* U-NET has an encoder-decoder architecture allowing the generation of noise.  
* U-NET employs skip-connections to ensure gradient propagation during back-propagation.
* [Diffusion models](https://arxiv.org/abs/2006.11239) embed the index of the time-step of
the DDIM process.
* The last convolution's kernel is initialized to all zeros, the mean of its targets. This practice improves the behavior at the start of training and make the mean squared error loss start at exactly 1.0.

Execute this code to instantiate these functions.   

In [ ]:

@keras.saving.register_keras_serializable()
def sinusoidal_embedding(x):
    embedding_min_frequency = 1.0
    frequencies = ops.exp(
        ops.linspace(
            ops.log(embedding_min_frequency),
            ops.log(embedding_max_frequency),
            embedding_dims // 2,
        )
    )
    angular_speeds = ops.cast(2.0 * math.pi * frequencies, "float32")
    embeddings = ops.concatenate(
        [ops.sin(angular_speeds * x), ops.cos(angular_speeds * x)], axis=3
    )
    return embeddings


def ResidualBlock(width):
    def apply(x):
        input_width = x.shape[3]
        if input_width == width:
            residual = x
        else:
            residual = layers.Conv2D(width, kernel_size=1)(x)
        x = layers.BatchNormalization(center=False, scale=False)(x)
        x = layers.Conv2D(width, kernel_size=3, padding="same", activation="swish")(x)
        x = layers.Conv2D(width, kernel_size=3, padding="same")(x)
        x = layers.Add()([x, residual])
        return x

    return apply


def DownBlock(width, block_depth):
    def apply(x):
        x, skips = x
        for _ in range(block_depth):
            x = ResidualBlock(width)(x)
            skips.append(x)
        x = layers.AveragePooling2D(pool_size=2)(x)
        return x

    return apply


def UpBlock(width, block_depth):
    def apply(x):
        x, skips = x
        x = layers.UpSampling2D(size=2, interpolation="bilinear")(x)
        for _ in range(block_depth):
            x = layers.Concatenate()([x, skips.pop()])
            x = ResidualBlock(width)(x)
        return x

    return apply


def get_network(image_size, widths, block_depth):
    noisy_images = keras.Input(shape=(image_size, image_size, 3))
    noise_variances = keras.Input(shape=(1, 1, 1))

    e = layers.Lambda(sinusoidal_embedding, output_shape=(1, 1, 32))(noise_variances)
    e = layers.UpSampling2D(size=image_size, interpolation="nearest")(e)

    x = layers.Conv2D(widths[0], kernel_size=1)(noisy_images)
    x = layers.Concatenate()([x, e])

    skips = []
    for width in widths[:-1]:
        x = DownBlock(width, block_depth)([x, skips])

    for _ in range(block_depth):
        x = ResidualBlock(widths[-1])(x)

    for width in reversed(widths[:-1]):
        x = UpBlock(width, block_depth)([x, skips])

    x = layers.Conv2D(3, kernel_size=1, kernel_initializer="zeros")(x)

    return keras.Model([noisy_images, noise_variances], x, name="residual_unet")


> **Exercise 7-2:** The foregoing code defines a relatively complex U-Net algorithm. In one, or a few, sentences explain why the DDIM algorithm requires positional embedding for the sampling.         

> **Answer:**        

## Diffusion model

With a few key components defined we are ready to define the complete `DiffusionModel` class.  

### 1. Diffusion scheduling

The diffusion process runs from time = 0, and ends at time = 1.0. Treating the diffusion time as a continuous variable allows the number of sampling steps to change at inference time. The diffusion schedule determines the noise levels and signal levels of the noisy image at each time step. The schedule funciton outputs two quantities: the `noise_rate` and the `signal_rate`
(corresponding to $\sqrt{1 - alpha}$ and $\sqrt{alpha}$. The noisy image is generated by weighting the random noise and the training image by the corresponding rates and adding them together.

The standard normal random noise and the normalized images both have zero mean and unit variance. The noise rate and signal rate can be interpreted as the standard deviations of their components in the noisy image. The square of these rates can be interpreted as the variance. The rates will always be set so that their squared sums to 1.0. As a result, the noisy images will always have unit variance, just like its unscaled components.

Here use a simplified, continuous version of a [cosine schedule (Section 3.2)](https://arxiv.org/abs/2102.09672). This schedule is symmetric, slow towards the start and end of the diffusion process and rapid in between. There is a nice geometric interpretation for this schedule, using the [trigonometric properties of the unit circle](https://en.wikipedia.org/wiki/Unit_circle#/media/File:Circle-trig6.svg):

![diffusion schedule gif](https://i.imgur.com/JW9W0fA.gif)

### 2. Sampling for reverse diffusion

When sampling the previous estimate of the noisy image is separated into image and noise using U-NET. These components are recombined using the signal and noise rate for the following step.

The code below implements the deterministic DDIM sampling procedure, which corresponds to $\eta = 0$ in the paper original. 

### 2. Training process

The training procedure follows these steps:      
1. The random diffusion times are uniformly sampled.
2. The training images are mixed with with random gaussian noises at rates according to the diffusion schedule.
3. The model learns to separate the noisy image to the image and signal components.

Typically, a neural network is trained to predict the unscaled noise component, from which the predicted image component can be calculated using the signal and noise rates. Theoretically, one would expect that the pixelwise [mean squared error](https://keras.io/api/losses/regression_losses/#mean_squared_error-function) to be used. To improve robustness
[mean absolute error](https://keras.io/api/losses/regression_losses/#mean_absolute_error-function)
is used instead.

Execute the code in the cell below to load the `DiffusionModel` class. 

In [ ]:

@keras.saving.register_keras_serializable()
class DiffusionModel(keras.Model):
    def __init__(self, image_size, widths, block_depth):
        super().__init__()

        self.normalizer = layers.Normalization()
        self.network = get_network(image_size, widths, block_depth)
        self.ema_network = keras.models.clone_model(self.network)

    def compile(self, **kwargs):
        super().compile(**kwargs)

        self.noise_loss_tracker = keras.metrics.Mean(name="n_loss")
        self.image_loss_tracker = keras.metrics.Mean(name="i_loss")
        self.kid = KID(name="kid")

    @property
    def metrics(self):
        return [self.noise_loss_tracker, self.image_loss_tracker, self.kid]

    def denormalize(self, images):
        # convert the pixel values back to 0-1 range
        images = self.normalizer.mean + images * self.normalizer.variance**0.5
        return ops.clip(images, 0.0, 1.0)

    def diffusion_schedule(self, diffusion_times):
        # diffusion times -> angles
        start_angle = ops.cast(ops.arccos(max_signal_rate), "float32")
        end_angle = ops.cast(ops.arccos(min_signal_rate), "float32")

        diffusion_angles = start_angle + diffusion_times * (end_angle - start_angle)

        # angles -> signal and noise rates
        signal_rates = ops.cos(diffusion_angles)
        noise_rates = ops.sin(diffusion_angles)
        # note that their squared sum is always: sin^2(x) + cos^2(x) = 1

        return noise_rates, signal_rates

    def denoise(self, noisy_images, noise_rates, signal_rates, training):
        # the exponential moving average weights are used at evaluation
        if training:
            network = self.network
        else:
            network = self.ema_network

        # predict noise component and calculate the image component using it
        pred_noises = network([noisy_images, noise_rates**2], training=training)
        pred_images = (noisy_images - noise_rates * pred_noises) / signal_rates

        return pred_noises, pred_images

    def reverse_diffusion(self, initial_noise, diffusion_steps):
        # reverse diffusion = sampling
        num_images = initial_noise.shape[0]
        step_size = 1.0 / diffusion_steps

        # important line:
        # at the first sampling step, the "noisy image" is pure noise
        # but its signal rate is assumed to be nonzero (min_signal_rate)
        next_noisy_images = initial_noise
        for step in range(diffusion_steps):
            noisy_images = next_noisy_images

            # separate the current noisy image to its components
            diffusion_times = ops.ones((num_images, 1, 1, 1)) - step * step_size
            noise_rates, signal_rates = self.diffusion_schedule(diffusion_times)
            pred_noises, pred_images = self.denoise(
                noisy_images, noise_rates, signal_rates, training=False
            )
            # network used in eval mode

            # remix the predicted components using the next signal and noise rates
            next_diffusion_times = diffusion_times - step_size
            next_noise_rates, next_signal_rates = self.diffusion_schedule(
                next_diffusion_times
            )
            next_noisy_images = (
                next_signal_rates * pred_images + next_noise_rates * pred_noises
            )
            # this new noisy image will be used in the next step

        return pred_images

    def generate(self, num_images, diffusion_steps):
        # noise -> images -> denormalized images
        initial_noise = keras.random.normal(
            shape=(num_images, image_size, image_size, 3)
        )
        generated_images = self.reverse_diffusion(initial_noise, diffusion_steps)
        generated_images = self.denormalize(generated_images)
        return generated_images

    def train_step(self, images):
        # normalize images to have standard deviation of 1, like the noises
        images = self.normalizer(images, training=True)
        noises = keras.random.normal(shape=(batch_size, image_size, image_size, 3))

        # sample uniform random diffusion times
        diffusion_times = keras.random.uniform(
            shape=(batch_size, 1, 1, 1), minval=0.0, maxval=1.0
        )
        noise_rates, signal_rates = self.diffusion_schedule(diffusion_times)
        # mix the images with noises accordingly
        noisy_images = signal_rates * images + noise_rates * noises

        with tf.GradientTape() as tape:
            # train the network to separate noisy images to their components
            pred_noises, pred_images = self.denoise(
                noisy_images, noise_rates, signal_rates, training=True
            )

            noise_loss = self.loss(noises, pred_noises)  # used for training
            image_loss = self.loss(images, pred_images)  # only used as metric

        gradients = tape.gradient(noise_loss, self.network.trainable_weights)
        self.optimizer.apply_gradients(zip(gradients, self.network.trainable_weights))

        self.noise_loss_tracker.update_state(noise_loss)
        self.image_loss_tracker.update_state(image_loss)

        # track the exponential moving averages of weights
        for weight, ema_weight in zip(self.network.weights, self.ema_network.weights):
            ema_weight.assign(ema * ema_weight + (1 - ema) * weight)

        # KID is not measured during the training phase for computational efficiency
        return {m.name: m.result() for m in self.metrics[:-1]}

    def test_step(self, images):
        # normalize images to have standard deviation of 1, like the noises
        images = self.normalizer(images, training=False)
        noises = keras.random.normal(shape=(batch_size, image_size, image_size, 3))

        # sample uniform random diffusion times
        diffusion_times = keras.random.uniform(
            shape=(batch_size, 1, 1, 1), minval=0.0, maxval=1.0
        )
        noise_rates, signal_rates = self.diffusion_schedule(diffusion_times)
        # mix the images with noises accordingly
        noisy_images = signal_rates * images + noise_rates * noises

        # use the network to separate noisy images to their components
        pred_noises, pred_images = self.denoise(
            noisy_images, noise_rates, signal_rates, training=False
        )

        noise_loss = self.loss(noises, pred_noises)
        image_loss = self.loss(images, pred_images)

        self.image_loss_tracker.update_state(image_loss)
        self.noise_loss_tracker.update_state(noise_loss)

        # measure KID between real and generated images
        # this is computationally demanding, kid_diffusion_steps has to be small
        images = self.denormalize(images)
        generated_images = self.generate(
            num_images=batch_size, diffusion_steps=kid_diffusion_steps
        )
        self.kid.update_state(images, generated_images)

        return {m.name: m.result() for m in self.metrics}

    def plot_images(self, epoch=None, logs=None, num_rows=3, num_cols=6):
        # plot random generated images for visual evaluation of generation quality
        generated_images = self.generate(
            num_images=num_rows * num_cols,
            diffusion_steps=plot_diffusion_steps,
        )

        plt.figure(figsize=(num_cols * 2.0, num_rows * 2.0))
        for row in range(num_rows):
            for col in range(num_cols):
                index = row * num_cols + col
                plt.subplot(num_rows, num_cols, index + 1)
                plt.imshow(generated_images[index])
                plt.axis("off")
        plt.tight_layout()
        plt.show()
        plt.close()

    def call(self, inputs, training=False):
        ## Method for the forward pass of the diffusion model
        noisy_images, noise_rates = inputs
        _, signal_rates = self.diffusion_schedule(noise_rates)  # Calculate signal rates from noise rates
        ## Return the denoised result
        return self.denoise(noisy_images, noise_rates, signal_rates, training=training)


> **Exercise 7-3:** The `DiffusionModel` class does the heavy-lifting for the DDIM image generation algorithm. In one or a few sentences, answer the following questions.    
> 1. Examine the code for the `diffusion_schedule` method examine the code in the `diffusion_schedule` method. Also, read the discussion in the diffusion schedule subsection of the text. Describe the noise schedule used for the forward diffusion process.  
> 2. Examine the code in the `denoise` method. This method is called at each reverse diffusion step in the `reverse_diffusion` method. Notice the if statement that switches between training the neural network and sampling. Answer the following questions:    
>    a. During training, what component of the diffusion model is the neural network learning to predict?    
>    b. How is the predicted noise used in the denoising proceses.   
>    c. How are the signal rate and noise rates computed by the `diffusion_schedule` method used to update the noise image?
> 3. The sampling schedule in the `train_step` is the key difference between the DDIM and DDPM models. Explain how this sampling is performed in the forward steps of the random diffusion process?  

> **Answers:**  
> 1.            
> 2. Answer to second question.     
>    a.               
>    b.                     
>    c.              
> 3.          

## Training

The code in the cell below trains the DDIM model to generate flower images using the  Oxford Flowers 102 dataset. Execute this code and examine the results as the training progresses.  

In [ ]:
# create and compile the model
model = DiffusionModel(image_size, widths, block_depth)
model.compile(
    optimizer=keras.optimizers.AdamW(
        learning_rate=learning_rate, weight_decay=weight_decay
    ),
    loss=keras.losses.mean_absolute_error,
)
# pixelwise mean absolute error is used as loss

# Build the model by calling it on a batch of data
# This will initialize the weights and allow the model to be saved
for images in train_dataset.take(1):
    # Generate random noise rates for the batch of images
    noise_rates = tf.random.uniform(shape=(batch_size, 1, 1, 1), minval=0.0, maxval=1.0)

    # Call the model with both images and noise rates
    model([images, noise_rates])

# save the best model based on the validation KID metric
checkpoint_path = "checkpoints/diffusion_model.weights.h5"
checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_weights_only=True,
    monitor="val_kid",
    mode="min",
    save_best_only=True,
)



# calculate mean and variance of training dataset for normalization
model.normalizer.adapt(train_dataset)

## Build the model
model.build(input_shape=(None,64,64,3))

# run training and plot generated images periodically
train_history = model.fit(
    train_dataset,
    epochs=num_epochs,
    validation_data=val_dataset,
    callbacks=[
        keras.callbacks.LambdaCallback(on_epoch_end=model.plot_images),
        checkpoint_callback,
    ],
)


> **Exercise 7-4:** Examine the code in the **Training** code cell. In brief, what loss function is used, and what error is being learned?     

> **Answer:**           

> **Exercise 7-5:** Examine the squence of generated images produced in the history of the training epochs. These images represent the end product of the denoising Markov chain.
> 1. What do the images produced in the first few epochs tell you about the learning of the denoising process parameters?
> 2. Given the quality of the images in the last few epochs what can you say about the neural network's ability to learn the noise parameters, $\epsilon_\theta$?      
> **End of exercise.**

> **Answers:**        
> 1.          
> 2.           

The next step is to evaluate the trajectory of the model training. Execute the code below to display the plots. 

In [ ]:
def plot_image_loss(history, ax=None, figsize=(5, 4)):
    '''Function to plot the image loss vs. epoch'''
    if ax is None:
        _, ax = plt.subplots(figsize=figsize)
    train_i_loss = history.history['i_loss']
    val_i_loss = history.history['val_i_loss']
    x = list(range(1, len(val_i_loss) + 1))
    ax.plot(x, val_i_loss, color = 'red', label = 'Validation image loss')
    ax.plot(x, train_i_loss, label = 'Train image losss')
    ax.legend()
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Image Loss')
    ax.set_title('Image Loss vs. Epoch')
    plt.show()

def plot_noise_loss(history, ax=None, figsize=(5, 4)):
    '''Function to plot the noise loss vs. epoch'''
    if ax is None:
        _, ax = plt.subplots(figsize=figsize)
    train_n_loss = history.history['n_loss']
    val_n_loss = history.history['val_n_loss']
    x = list(range(1, len(val_n_loss) + 1))
    ax.plot(x, val_n_loss, color = 'red', label = 'Validation noise loss')
    ax.plot(x, train_n_loss, label = 'Train noise losss')
    ax.legend()
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Noise Loss')
    ax.set_title('Noise Loss vs. Epoch')
    plt.show()

def plot_kid(history,  ax=None, figsize=(5, 4)):
    '''Function to plot the noise loss vs. epoch'''
    if ax is None:
        _, ax = plt.subplots(figsize=figsize)
    kid = history.history['val_kid']
    x = list(range(1, len(kid) + 1))
    ax.plot(x, kid, label = 'Validatio KID')
    ax.legend()
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Validation KID')
    ax.set_title('Validation KID vs. Epoch')
    plt.show()

plot_noise_loss(train_history) #, ax=ax[1])
plot_image_loss(train_history) #, ax=ax[0])
plot_kid(train_history) #, ax=ax[2])

> **Exercise 7-6:** Examine the three plots produced from the training metrics and answer the following questions.     
> 1. Is the monotonic decrease in noise loss expected given the training algorithm and why?
> 2. Image loss measures the difference between the training imaage and the denoising image. This metric is not used to learn the model, but only for monitoring the training. Why does the general shape of the image loss curve closely resemble the noise loss curve?
> 3. The kernel inception distance (KDI) curve does not decrease monotonically. Is this behavior reasonable as the model learns and why?
> 4. Given the noise loss, image loss and KDI curves does it appear that the training of the model has largely converged?    

> **Answers:**      
> 1.      
> 2.        
> 3.      
> 4.           

## Inference

With the model trained and evaluated, its time to have some fun performing inference. Execute the code in the cell below.  

In [ ]:
# load the best model and generate images
model.load_weights(checkpoint_path)
model.plot_images()

> **Exercise 7-7:**  Examine the images produced by the best trained model in the Inference section of the notebook. All of the images are of floweres. However, there is no guidance indicating the images produced should be of flowers.    
> 1. Considering the training images, why do you think only flower images are created?
> 2. What are the implications of your answer to the above question in terms the distributions required for creating a diffusion image generative model that can create a wide range of images?
> 3. What are the limitations you have just discussed have for the general ability of generative diffusion models to produce arbitrary iamges in practice?       
> **End of exercise.**  

> **Answers:**
> 1.        
> 2.        
> 3.              

## Additional Results and Commentary

The final section of this notebook is largely unedited from the original version created by 

### Additional Examples 

By running the training for at least 50 epochs (takes 2 hours on a T4 GPU and 30 minutes
on an A100 GPU), one can get high quality image generations using this code example.

The evolution of a batch of images over a 80 epoch training (color artifacts are due to
GIF compression):

![flowers training gif](https://i.imgur.com/FSCKtZq.gif)

Images generated using between 1 and 20 sampling steps from the same initial noise:

![flowers sampling steps gif](https://i.imgur.com/tM5LyH3.gif)

Interpolation (spherical) between initial noise samples:

![flowers interpolation gif](https://i.imgur.com/hk5Hd5o.gif)

Deterministic sampling process (noisy images on top, predicted images on bottom, 40
steps):

![flowers deterministic generation gif](https://i.imgur.com/wCvzynh.gif)

Stochastic sampling process (noisy images on top, predicted images on bottom, 80 steps):

![flowers stochastic generation gif](https://i.imgur.com/kRXOGzd.gif)

### Lessons learned

During preparation for this code example I have run numerous experiments using
[this repository](https://github.com/beresandras/clear-diffusion-keras).
In this section I list
the lessons learned and my recommendations in my subjective order of importance.

### Algorithmic tips

* **min. and max. signal rates**: I found the min. signal rate to be an important
hyperparameter. Setting it too low will make the generated images oversaturated, while
setting it too high will make them undersaturated. I recommend tuning it carefully. Also,
setting it to 0 will lead to a division by zero error. The max. signal rate can be set to
1, but I found that setting it lower slightly improves generation quality.
* **loss function**: While large models tend to use mean squared error (MSE) loss, I
recommend using mean absolute error (MAE) on this dataset. In my experience MSE loss
generates more diverse samples (it also seems to hallucinate more
[Section 3](https://arxiv.org/abs/2111.05826)), while MAE loss leads to smoother images.
I recommend trying both.
* **weight decay**: I did occasionally run into diverged trainings when scaling up the
model, and found that weight decay helps in avoiding instabilities at a low performance
cost. This is why I use
[AdamW](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/experimental/AdamW)
instead of [Adam](https://keras.io/api/optimizers/adam/) in this example.
* **exponential moving average of weights**: This helps to reduce the variance of the KID
metric, and helps in averaging out short-term changes during training.
* **image augmentations**: Though I did not use image augmentations in this example, in
my experience adding horizontal flips to the training increases generation performance,
while random crops do not. Since we use a supervised denoising loss, overfitting can be
an issue, so image augmentations might be important on small datasets. One should also be
careful not to use
[leaky augmentations](https://keras.io/examples/generative/gan_ada/#invertible-data-augmentation),
which can be done following
[this method (end of Section 5)](https://arxiv.org/abs/2206.00364) for instance.
* **data normalization**: In the literature the pixel values of images are usually
converted to the -1 to 1 range. For theoretical correctness, I normalize the images to
have zero mean and unit variance instead, exactly like the random noises.
* **noise level input**: I chose to input the noise variance to the network, as it is
symmetrical under our sampling schedule. One could also input the noise rate (similar
performance), the signal rate (lower performance), or even the
[log-signal-to-noise ratio (Appendix B.1)](https://arxiv.org/abs/2107.00630)
(did not try, as its range is highly
dependent on the min. and max. signal rates, and would require adjusting the min.
embedding frequency accordingly).
* **gradient clipping**: Using global gradient clipping with a value of 1 can help with
training stability for large models, but decreased performance significantly in my
experience.
* **residual connection downscaling**: For
[deeper models (Appendix B)](https://arxiv.org/abs/2205.11487), scaling the residual
connections with 1/sqrt(2) can be helpful, but did not help in my case.
* **learning rate**: For me, [Adam optimizer's](https://keras.io/api/optimizers/adam/)
default learning rate of 1e-3 worked very well, but lower learning rates are more common
in the [literature (Tables 11-13)](https://arxiv.org/abs/2105.05233).

### Architectural tips

* **sinusoidal embedding**: Using sinusoidal embeddings on the noise level input of the
network is crucial for good performance. I recommend setting the min. embedding frequency
to the reciprocal of the range of this input, and since we use the noise variance in this
example, it can be left always at 1. The max. embedding frequency controls the smallest
change in the noise variance that the network will be sensitive to, and the embedding
dimensions set the number of frequency components in the embedding. In my experience the
performance is not too sensitive to these values.
* **skip connections**: Using skip connections in the network architecture is absolutely
critical, without them the model will fail to learn to denoise at a good performance.
* **residual connections**: In my experience residual connections also significantly
improve performance, but this might be due to the fact that we only input the noise
level embeddings to the first layer of the network instead of to all of them.
* **normalization**: When scaling up the model, I did occasionally encounter diverged
trainings, using normalization layers helped to mitigate this issue. In the literature it
is common to use
[GroupNormalization](https://www.tensorflow.org/addons/api_docs/python/tfa/layers/GroupNormalization)
(with 8 groups for example) or
[LayerNormalization](https://keras.io/api/layers/normalization_layers/layer_normalization/)
in the network, I however chose to use
[BatchNormalization](https://keras.io/api/layers/normalization_layers/batch_normalization/),
as it gave similar benefits in my experiments but was computationally lighter.
* **activations**: The choice of activation functions had a larger effect on generation
quality than I expected. In my experiments using non-monotonic activation functions
outperformed monotonic ones (such as
[ReLU](https://www.tensorflow.org/api_docs/python/tf/keras/activations/relu)), with
[Swish](https://www.tensorflow.org/api_docs/python/tf/keras/activations/swish) performing
the best (this is also what [Imagen uses, page 41](https://arxiv.org/abs/2205.11487)).
* **attention**: As mentioned earlier, it is common in the literature to use
[attention layers](https://keras.io/api/layers/attention_layers/multi_head_attention/) at low
resolutions for better global coherence. I omitted them for simplicity.
* **upsampling**:
[Bilinear and nearest neighbour upsampling](https://keras.io/api/layers/reshaping_layers/up_sampling2d/)
in the network performed similarly, however I did not try
[transposed convolutions](https://keras.io/api/layers/convolution_layers/convolution2d_transpose/).

For a similar list about GANs check out
[this Keras tutorial](https://keras.io/examples/generative/gan_ada/#gan-tips-and-tricks).

#### Copyright 2026, Stephen F Elston. All rights reserved. 